<pre style="font-size: 15px; line-height: 1.2;">
---
course_code: 02BMSBI24365
course_title: AI and ML in Bioinformatics
institution: Garden City University (GCU)
program: M.Sc. Bioinformatics (Semester III)
unit: Unit 1 — Foundations of Machine Learning for Bioinformatics
submodule: 1.2 Data Preprocessing & Feature Engineering
document_type: Guided Lecture Notebook (Part A)
ai_tier: Full AI (Learning Aid)
---
</pre>

# Data Preprocessing (Part A): Structural Integrity & Representation

**Objective:** Before we can train machine learning algorithms on high-dimensional biological data (like $20,000$ gene expression profiles), we must first master the mechanics of data cleaning. 

In this Part A session, we will isolate and solve three fundamental data problems using universally understood, intuitive datasets (House Prices and Apples vs. Oranges). Once these mechanics are mastered, we will apply them to real bioinformatics datasets in Part B.

### Today's Focus:
1. **Structural Problems:** Missing values, non-fitting text, and duplicate records.
2. **Representation Problems:** Translating text categories into mathematical tensors (One-Hot Encoding).
3. **Dimensionality Problems:** Identifying and removing zero-variance (useless) features.

## 1. Code Environment Setup
We begin by importing the standard data science stack. We fix our random seed to ensure computational reproducibility.

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import VarianceThreshold


# Set reproducibility seed
np.random.seed(42)
# Pandas display options
pd.set_option('display.max_columns', None)

## 2. Generating the Messy Toy Datasets
We will synthesize two heavily corrupted "messy" datasets containing all the standard errors one might find in raw data.

In [4]:
# 1. Messy House Price Dataset
messy_houses = pd.DataFrame({
    'House_ID': [101, 102, 103, 104, 105, 103],          # Notice House 103 is duplicated
    'Square_Feet': [1500, 2200, 1800, 2500, 1900, 1800],
    'Bedrooms': [3, np.nan, 3, 4, 3, 3],                 # Notice the missing value (NaN)
    'Has_Roof': [1, 1, 1, 1, 1, 1],                      # Notice this never changes (Zero Variance)
    'Price_USD': [300000, 450000, 'Error', 500000, np.inf, 'Error']  # Text and Infinity in a numeric column
})

# 2. Messy Fruit Dataset
messy_fruits = pd.DataFrame({
    'Fruit_ID': [1, 2, 3, 4, 5],
    'Weight_Grams': [150, 130, 160, 140, 155],
    'Color': ['Red', 'Orange', 'Green', 'Red', 'Orange'], # Text categories that ML cannot read
    'Target_Class': ['Apple', 'Orange', 'Apple', 'Apple', 'Orange']
})

print("--- Messy House Dataset ---")
display(messy_houses)

print("\n--- Messy Fruit Dataset ---")
display(messy_fruits)

--- Messy House Dataset ---


,House_ID,Square_Feet,Bedrooms,Has_Roof,Price_USD
0,101,1500,3.0,1,300000
1,102,2200,NaN,1,450000
2,103,1800,3.0,1,Error
3,104,2500,4.0,1,500000
4,105,1900,3.0,1,inf
5,103,1800,3.0,1,Error



--- Messy Fruit Dataset ---


,Fruit_ID,Weight_Grams,Color,Target_Class
0,1,150,Red,Apple
1,2,130,Orange,Orange
2,3,160,Green,Apple
3,4,140,Red,Apple
4,5,155,Orange,Orange


---
## Topic 1: Structural & Formatting Problems
Raw data is rarely clean. Sensors fail, humans make data-entry errors, and databases merge incorrectly. Let's fix the structural integrity of `messy_houses`.

### Step 1A: Duplicate Records
If a model sees the exact same data point twice, it artificially inflates the model's confidence in that specific pattern.

In [5]:
# Remove exact duplicate rows
clean_houses = messy_houses.drop_duplicates()

print(f"Rows before: {len(messy_houses)} | Rows after removing duplicates: {len(clean_houses)}")
display(clean_houses)

Rows before: 6 | Rows after removing duplicates: 5


,House_ID,Square_Feet,Bedrooms,Has_Roof,Price_USD
0,101,1500,3.0,1,300000
1,102,2200,NaN,1,450000
2,103,1800,3.0,1,Error
3,104,2500,4.0,1,500000
4,105,1900,3.0,1,inf


### Step 1B: Non-Fitting / Invalid Values
Machine learning math crashes if it encounters text (like `'Error'`) or mathematical impossibilities (like `np.inf`) inside a column that should only contain numbers.

In [6]:
# Force the 'Price_USD' column to be numeric. Any text like 'Error' will be coerced into NaN (Missing)
clean_houses['Price_USD'] = pd.to_numeric(clean_houses['Price_USD'], errors='coerce')

# Replace Infinity (np.inf) with NaN
clean_houses.replace([np.inf, -np.inf], np.nan, inplace=True)

print("After coercing errors and infinities to NaN:")
display(clean_houses)

After coercing errors and infinities to NaN:


/var/folders/w2/23fl2tz9739cd6h0_pk4nl0r0000gn/T/ipykernel_47008/2128242296.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_houses['Price_USD'] = pd.to_numeric(clean_houses['Price_USD'], errors='coerce')
/var/folders/w2/23fl2tz9739cd6h0_pk4nl0r0000gn/T/ipykernel_47008/2128242296.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_houses.replace([np.inf, -np.inf], np.nan, inplace=True)


,House_ID,Square_Feet,Bedrooms,Has_Roof,Price_USD
0,101,1500,3.0,1,300000.0
1,102,2200,NaN,1,450000.0
2,103,1800,3.0,1,NaN
3,104,2500,4.0,1,500000.0
4,105,1900,3.0,1,NaN


### Step 1C: Missing Values (`NaN`)
Now that we have converted our errors into standard `NaN` (Not a Number) values, we must decide how to handle the "holes" in our dataset.

We use **Imputation** to intelligently guess the missing values based on the distribution of the known data.

In [7]:
# Initialize a SimpleImputer to replace NaN with the 'median' of the column
imputer = SimpleImputer(strategy='median')

# We only impute the numeric columns that have missing data
columns_to_impute = ['Bedrooms', 'Price_USD']

clean_houses[columns_to_impute] = imputer.fit_transform(clean_houses[columns_to_impute])

print("After Median Imputation:")
display(clean_houses)

After Median Imputation:


/var/folders/w2/23fl2tz9739cd6h0_pk4nl0r0000gn/T/ipykernel_47008/4173866244.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_houses[columns_to_impute] = imputer.fit_transform(clean_houses[columns_to_impute])


,House_ID,Square_Feet,Bedrooms,Has_Roof,Price_USD
0,101,1500,3.0,1,300000.0
1,102,2200,3.0,1,450000.0
2,103,1800,3.0,1,450000.0
3,104,2500,4.0,1,500000.0
4,105,1900,3.0,1,450000.0


---
## Topic 2: Dimensionality & Zero-Variance Features
If a feature is exactly the same for every single row, it is useless for Machine Learning.

In our dataset, `Has_Roof` is `1` for every single house. If every house has a roof, knowing that a house has a roof doesn't help us predict why it is more expensive than another.

We can easily identify and remove these "Zero-Variance" features.

In [8]:
# 1. Initialize the tool to drop columns with 0 variance
selector = VarianceThreshold(threshold=0.0)

# 2. Tell scikit-learn to return a Pandas DataFrame (keeps our column names!)
selector.set_output(transform="pandas")

# 3. Apply it to our numerical features
numerical_features = ['Square_Feet', 'Bedrooms', 'Has_Roof', 'Price_USD']
final_houses = selector.fit_transform(clean_houses[numerical_features])

print("After Zero-Variance Filtering (Notice 'Has_Roof' is automatically dropped):")
display(final_houses)

After Zero-Variance Filtering (Notice 'Has_Roof' is automatically dropped):


,Square_Feet,Bedrooms,Price_USD
0,1500,3.0,300000.0
1,2200,3.0,450000.0
2,1800,3.0,450000.0
3,2500,4.0,500000.0
4,1900,3.0,450000.0


---
## Topic 3: Representation Problems (The Language Barrier)
An ML algorithm is just a massive calculator. It cannot multiply $5 \times \text{"Red"}$. If we feed the `messy_fruits` dataset directly into a model, it will crash. We must translate text categories into mathematical tensors using **One-Hot Encoding**.

In [11]:
print("Original Fruit Data:")
display(messy_fruits)

# Use Pandas get_dummies to perform One-Hot Encoding on the 'Color' column
encoded_fruits = pd.get_dummies(messy_fruits, columns=['Color'], dtype=int)

print("\nAfter One-Hot Encoding (Notice the new binary indicator columns):")
display(encoded_fruits)

Original Fruit Data:


,Fruit_ID,Weight_Grams,Color,Target_Class
0,1,150,Red,Apple
1,2,130,Orange,Orange
2,3,160,Green,Apple
3,4,140,Red,Apple
4,5,155,Orange,Orange



After One-Hot Encoding (Notice the new binary indicator columns):


,Fruit_ID,Weight_Grams,Target_Class,Color_Green,Color_Orange,Color_Red
0,1,150,Apple,0,0,1
1,2,130,Orange,0,1,0
2,3,160,Apple,1,0,0
3,4,140,Apple,0,0,1
4,5,155,Orange,0,1,0


---
## 🧪 Student Challenge: Clinical Cohort Cleanup
**Your turn.** You have just received a tiny clinical dataset from the hospital (`clinical_data`). It is riddled with errors.

Your objective is to execute the 4 core steps learned today:
1. Drop the duplicate patient row.
2. Coerce the `White_Blood_Cell_Count` text error to `NaN`, then use `SimpleImputer(strategy='mean')` to fill all missing values.
3. Use `pd.get_dummies()` to One-Hot Encode the `Gender` column.
4. Drop the `Is_Human` column manually, as it possesses zero biological variance.

In [10]:
# The Messy Clinical Dataset
clinical_data = pd.DataFrame({
    'Patient_ID': ['P1', 'P2', 'P3', 'P4', 'P3', 'P6'],
    'White_Blood_Cell_Count': [4500, np.nan, 8000, 6200, 8000, 'Error_Read'],
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Female', 'Male'],
    'Is_Human': [1, 1, 1, 1, 1, 1]
})

print("--- Raw Clinical Data ---")
display(clinical_data)

# ==========================================
# YOUR CODE HERE
# ==========================================




--- Raw Clinical Data ---


,Patient_ID,White_Blood_Cell_Count,Gender,Is_Human
0,P1,4500,Male,1
1,P2,NaN,Female,1
2,P3,8000,Female,1
3,P4,6200,Male,1
4,P3,8000,Female,1
5,P6,Error_Read,Male,1
